# Story compositions — a face-locked character across a multi-axis comic (flux-recipes)
An extensive compositional demo on the **batch** (story) run path. We **generate our own references** with base FLUX
(no ops) — a character face, a garment, a material palette — so the notebook is self-contained. Then we run the SAME
scene list through three composition **axes**, each routed to the specialist that owns it:

1. **identity** — `identity_story` (PuLID face-lock): one character across scenes.
2. **material** — `identity_story_material` (step-gated K/V-share): one shared palette across every panel.
3. **garment** — **CatVTON** try-on: the character actually WEARING a specific garment (Redux can't — it clones a
   product-shot; a worn garment is a grounded operand a try-on model produces, not an appearance-channel effect).

Axes 1–2 are one FLUX.1-dev denoise; axis 3 is a second-stage FLUX.1-Fill edit that touches only the clothing region
(face untouched). We run the generator axes first, then free it and load CatVTON — no model thrash. Metrics: ArcFace
(identity holds) + CLIP-to-reference (palette/garment reads). Metrics rank, eyeball decides. OPEN-WEIGHT. 80GB A100.

In [ ]:
import subprocess, os
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image
!pip install -q insightface facexlib onnxruntime-gpu timm einops ftfy opencv-python-headless peft
subprocess.run(["rm","-rf","flux-recipes"])
subprocess.run(["git","clone","-q","-b","main","https://github.com/remyxai/flux-recipes.git"])

In [ ]:
import sys, torch, numpy as np, cv2, gc
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"flux-recipes")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
assert torch.cuda.is_available(); print("GPU:",torch.cuda.get_device_name(0))
from flux_modular import RecipeRunner
runner=RecipeRunner(steps=20)
from transformers import CLIPModel, CLIPProcessor
from PIL import Image, ImageDraw
from IPython.display import display
_clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda").eval(); _cp=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_img(a,b):
    px=_cp(images=[a,b],return_tensors="pt").to("cuda"); v=_clip.vision_model(pixel_values=px["pixel_values"]).pooler_output
    e=_clip.visual_projection(v); e=e/e.norm(dim=-1,keepdim=True); return float((e[0]@e[1]).cpu())
def fmt(x): return f"{x:.2f}" if x is not None else "no-face"
def strip(tiles, cell=300, sub=True):
    n=len(tiles); g=Image.new("RGB",(n*cell+(n+1)*8, cell+40),"white"); d=ImageDraw.Draw(g)
    for j,(cap,im,s) in enumerate(tiles):
        x=8+j*(cell+8); g.paste(im.convert("RGB").resize((cell,cell)),(x,4)); d.text((x+4,cell+8),str(cap)[:40],fill="black")
        if sub and s: d.text((x+4,cell+22),str(s)[:40],fill="black")
    display(g)
print("runner + CLIP ready")

## Generate references with base FLUX (no ops) — a character face, a garment, a palette
`{'run':'default'}` with no capture/condition/ops is a plain text2img through the same components. Fixed seeds.

In [ ]:
BASE={"name":"base","run":"default","inputs":["prompt"],"params":{"guidance":3.5}}
def gen(p, s): return runner.run(BASE, {"prompt":p}, seed=s)
FACE = gen("a studio portrait headshot of a woman with short auburn hair and green eyes, freckles, neutral background, sharp focus", 7)
JACKET = gen("a worn brown leather aviator jacket with a fur collar, product photo on white background, front view", 11)
PALETTE = gen("an abstract field of warm teal and amber stained glass, rich saturated color texture", 21)
strip([("FACE (identity)",FACE,""),("JACKET (garment)",JACKET,""),("PALETTE (material)",PALETTE,"")], sub=False)
print("references: FACE = the character; JACKET = the garment axis; PALETTE = the material axis.")

## The scene list — clear, UNOCCLUDED half-body framing (the CatVTON auto-mask needs a visible torso)
Hands at the sides, no object held across the chest. A held book/letter or a foreground occluder confuses the
segformer clothes-mask and the garment axis renders a generic top instead of the jacket (measured — see the caveat
below axis 3).

In [ ]:
SCENES=["a half-body portrait standing by a tall window in a lamplit study, hands at her sides",
        "a half-body portrait standing on a windswept clifftop at dawn, hands at her sides",
        "a half-body portrait standing in a bustling market square at noon, hands at her sides"]
print(len(SCENES),"panels")

## Axis 1 — `identity_story`: face-locked character across scenes (FLUX.1-dev)

In [ ]:
C1={"name":"identity_story","run":"batch","inputs":["id_image","scene_prompts"],"params":{"id_weight":1.0,"guidance":3.5}}
p_id=runner.run(C1, {"id_image":FACE,"scene_prompts":SCENES}, seed=0)
# ArcFace via the PuLID encoder (loaded by the identity run) — grab the InsightFace app now, keep it past the free
from flux_modular.identity import _PULID
_app=_PULID["enc"].app
def arc_emb(pil):
    fi=_app.get(cv2.cvtColor(np.asarray(pil.convert("RGB")),cv2.COLOR_RGB2BGR))
    if not fi: return None
    fi=sorted(fi,key=lambda x:(x['bbox'][2]-x['bbox'][0])*(x['bbox'][3]-x['bbox'][1]))[-1]
    e=fi['embedding']; return e/(np.linalg.norm(e)+1e-9)
_ref=arc_emb(FACE)
def arc(im):
    e=arc_emb(im); return float(np.dot(e,_ref)) if (e is not None and _ref is not None) else None
strip([("FACE",FACE,"")]+[(f"panel{i+1}", p_id[i], f"face={fmt(arc(p_id[i]))}") for i in range(len(p_id))])
print("axis 1: same character across scenes (undressed, neutral palette).")

## Axis 2 — `identity_story_material`: ONE shared palette across every panel (gated K/V, FLUX.1-dev)
The palette is captured once from PALETTE and appended LATE (`start_frac=0.4`) to every panel — content early
(character/scene), style late (palette) — so it art-directs the strip without swamping the face.

In [ ]:
C2={"name":"identity_story_material","run":"batch","inputs":["id_image","scene_prompts","ref_appearance"],
    "condition":{"kind":"kv_appearance","source":"ref_appearance","edge":2,"sigma":0.35,"timestep":661,"start_frac":0.4},
    "params":{"id_weight":1.0,"guidance":3.5}}
p_mat=runner.run(C2, {"id_image":FACE,"scene_prompts":SCENES,"ref_appearance":PALETTE}, seed=0)
strip([("PALETTE",PALETTE,"")]+[(f"panel{i+1}", p_mat[i], f"face={fmt(arc(p_mat[i]))} pal={clip_img(p_mat[i],PALETTE):.2f}") for i in range(len(p_mat))])
print("axis 2: the teal/amber palette applied consistently across panels; face should survive (arcface ~ axis 1).")
print("GO if palette reads on all three + face held; NO-GO if a panel swamps (no-face) or palette flat vs axis 1.")

## Free the generator, load CatVTON (FLUX.1-Fill) — the garment axis is a second-stage specialist

In [ ]:
del runner; gc.collect(); torch.cuda.empty_cache()
print("freed FLUX.1-dev; mem:", round(torch.cuda.memory_allocated()/1e9,1),"GB")
from diffusers import ModularPipeline
vton = ModularPipeline.from_pretrained("remyxai/catvton-flux-modular", trust_remote_code=True)
vton.load_components(dtype=torch.bfloat16); vton.to("cuda")
def dress(person, garment):
    g=torch.Generator("cuda").manual_seed(0)
    return vton(person_image=person, garment_image=garment, height=768, width=576,
                guidance_scale=30.0, num_inference_steps=30, generator=g).images[0]
print("CatVTON (FLUX.1-Fill + LoRA + segformer masker) loaded")

## Axis 3 — CatVTON: dress the character in the jacket (auto-mask, face untouched)
Applied to BOTH the plain identity panels (garment alone) and the palette panels (**palette + garment together** —
all three axes at once).

In [ ]:
p_gar  = [dress(p_id[i],  JACKET) for i in range(len(p_id))]   # identity + garment
p_full = [dress(p_mat[i], JACKET) for i in range(len(p_mat))]  # identity + palette + garment (full stack)
for i in range(len(p_id)):
    strip([("garment",JACKET,""),
           (f"panel{i+1} id-only", p_id[i], f"face={fmt(arc(p_id[i]))} jkt={clip_img(p_id[i],JACKET):.2f}"),
           (f"panel{i+1} DRESSED", p_gar[i], f"face={fmt(arc(p_gar[i]))} jkt={clip_img(p_gar[i],JACKET):.2f}")], cell=300)
print("axis 3: face kept through the edit (arcface ~ id-only — CatVTON edits only the torso) + the jacket is WORN")
print("(jkt-CLIP rises id-only->dressed). The worn garment Redux could not deliver — try-on specialist, staged.")

**Garment-axis caveat (measured).** CatVTON's segformer auto-mask needs a clear torso. When an object occludes
the chest (a held book/letter, a ship's wheel), the mask is partial and the try-on renders a generic top instead of
the jacket — so these scenes use hands-down standing framing. For an occluded pose, pass a manual `mask` or reframe.
Identity is preserved regardless (CatVTON edits only the clothing region); occlusion affects the *garment*, not the
face.

## The axis matrix — the same scene under each composition

In [ ]:
for i in range(len(SCENES)):
    strip([("identity",p_id[i],f"face={fmt(arc(p_id[i]))}"),
           ("+ material",p_mat[i],f"face={fmt(arc(p_mat[i]))} pal={clip_img(p_mat[i],PALETTE):.2f}"),
           ("+ garment",p_gar[i],f"face={fmt(arc(p_gar[i]))} jkt={clip_img(p_gar[i],JACKET):.2f}"),
           ("+ material + garment",p_full[i],f"face={fmt(arc(p_full[i]))}")], cell=300)
print("each row = one scene under: identity | +material palette | +garment | +both. Face persists across all four;")
print("each axis adds on top via its own specialist (PuLID face-lock / gated K/V palette / CatVTON try-on).")

## The finished comic — one character, one wardrobe, art-directed

In [ ]:
strip([("FACE",FACE,"")]+[(f"panel{i+1}", p_full[i], f"face={fmt(arc(p_full[i]))}") for i in range(len(p_full))], cell=340)
print("identity_story (face) + identity_story_material (palette) + CatVTON (garment) — three axes, three specialists.")